# Algoritmo de Grover

**Objetivo:** implementar el oráculo, el difusor y la amplificación de amplitudes paso a paso.

Grover resuelve búsqueda no estructurada en O(√N) vs O(N) clásico.
Para N=4 (2 qubits) solo necesitamos **1 iteración** para encontrar el elemento marcado.

In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

# Buscamos el elemento |11⟩ en un espacio de 4 elementos
# Oráculo: aplica fase -1 al estado marcado
def oracle_11():
    qc = QuantumCircuit(2)
    qc.cz(0, 1)   # aplica Z en |11⟩ → fase -1
    return qc

qc = oracle_11()
print(qc.draw())

# Verificar: |11⟩ → -|11⟩, el resto sin cambios
for label in ['00','01','10','11']:
    idx = int(label, 2)
    state = np.zeros(4); state[idx] = 1.0
    sv = Statevector(state)
    result = sv.evolve(qc)
    if abs(result.data[idx] + 1) < 0.01:
        print(f'|{label}⟩ → -|{label}⟩  ✓ (marcado)')
    else:
        print(f'|{label}⟩ → {result.data[idx]:.1f}|{label}⟩')

## El Difusor

El difusor amplifica la amplitud del estado marcado invirtiendo todas las amplitudes alrededor de la media:
D = 2|s⟩⟨s| − I,  donde |s⟩ = H⊗n|0⊗n⟩ es la superposición uniforme.

In [ ]:
def diffuser(n):
    qc = QuantumCircuit(n)
    qc.h(range(n))
    qc.x(range(n))
    qc.h(n-1)
    qc.mcx(list(range(n-1)), n-1)  # multi-controlled-X
    qc.h(n-1)
    qc.x(range(n))
    qc.h(range(n))
    return qc

print(diffuser(2).draw())

## Circuito Completo de Grover

Para N=4 elementos y 1 solución, el número óptimo de iteraciones es k ≈ π/4 · √N ≈ 1.

In [ ]:
n = 2
qc = QuantumCircuit(n)

# Estado inicial: superposición uniforme
qc.h(range(n))

# 1 iteración Grover = oráculo + difusor
qc.compose(oracle_11(), inplace=True)
qc.compose(diffuser(n), inplace=True)

print(qc.draw())

sv = Statevector.from_instruction(qc)
print('\nAmplitudes finales:')
for i, amp in enumerate(sv.data):
    label = format(i, f'0{n}b')
    print(f'  |{label}⟩: {amp:.4f}  P={abs(amp)**2:.3f}')

In [ ]:
# Simulación con medición
from qiskit import transpile
from qiskit_aer import AerSimulator

qc_med = qc.copy()
qc_med.measure_all()

sim = AerSimulator()
job = sim.run(transpile(qc_med, sim), shots=1000)
conteos = job.result().get_counts()
print('Conteos (1000 shots):', conteos)
print(f'Probabilidad |11⟩: {conteos.get("11",0)/1000:.3f} (esperado ≈ 1.0)')

## Escalado: N qubits

Para N qubits (2^N elementos), el número óptimo de iteraciones crece como √(2^N/k) donde k es el número de soluciones.

In [ ]:
import matplotlib.pyplot as plt

# Probabilidad de éxito tras k iteraciones (fórmula analítica)
# Para 1 solución en M=2^n elementos:
def prob_grover(k, M):
    theta = np.arcsin(1/np.sqrt(M))
    return np.sin((2*k+1)*theta)**2

fig, axes = plt.subplots(1, 2, figsize=(10,4))
for n_q, ax in zip([2, 4], axes):
    M = 2**n_q
    k_range = range(1, int(np.pi/4 * np.sqrt(M)) + 5)
    probs = [prob_grover(k, M) for k in k_range]
    ax.plot(list(k_range), probs, 'o-', color='steelblue')
    k_opt = int(np.pi/4 * np.sqrt(M))
    ax.axvline(k_opt, color='red', linestyle='--', label=f'k_opt={k_opt}')
    ax.set_title(f'n={n_q} qubits (M={M})')
    ax.set_xlabel('Iteraciones k'); ax.set_ylabel('P(éxito)')
    ax.legend(); ax.set_ylim(0, 1.1)
plt.tight_layout(); plt.savefig('grover_scaling.png', dpi=80); plt.show()
print('Guardado: grover_scaling.png')

## Ejercicio

1. Modifica el oráculo para buscar |01⟩ en vez de |11⟩.
2. ¿Cuántas iteraciones necesitas para N=8 qubits y 1 solución?
3. ¿Qué pasa si hay 2 soluciones en lugar de 1?

In [ ]:
# Tu solución — oráculo para |01⟩
def oracle_01():
    qc = QuantumCircuit(2)
    qc.x(1)        # invierte qubit 1 (01 → 11)
    qc.cz(0, 1)   # fase -1 en |11⟩
    qc.x(1)        # deshace inversión
    return qc

print(oracle_01().draw())

## Próximos pasos

- **Lab completo:** `Cuadernos/laboratorios/01_grover_dos_qubits.ipynb`
- **Módulo teórico:** `Tutorial/05_algoritmos/README.md`
- **Siguiente guía:** `04_teleportacion_cuantica.ipynb`